In [6]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")

Project root: /Users/matthewho/Documents/research/ctx_editor


In [7]:
# baseline LiC run: 2026-02-17/01-58-08
baseline_lic_run_dir = PROJECT_ROOT / "outputs/2026-02-17/01-58-08"

# baseline natural run: 2026-02-17/08-06-15
baseline_natural_run_dir = PROJECT_ROOT / "outputs/2026-02-17/08-06-15"

In [8]:
!ls {baseline_natural_run_dir}

config.yaml         metrics.json        run_experiment.log  traces/
experiment.log      results.json        summary.txt         verbose.log


In [9]:
natural_metrics = read_json(baseline_natural_run_dir / "metrics.json")

In [10]:
natural_metrics

{'experiment_name': 'baseline_natural_gpt-5-mini_t30d_all_tasks',
 'total_samples': 119,
 'total_attempted': 120,
 'errors': 1,
 'correct': 78,
 'accuracy': 0.6554621848739496,
 'average_score': 0.6554621848739496,
 'total_cost_usd': 0.9836416,
 'average_turns': 4.310924369747899,
 'execution_mode': 'parallel',
 'per_task': {'actions': {'total_samples': 30,
   'total_attempted': 30,
   'errors': 0,
   'correct': 18,
   'accuracy': 0.6,
   'average_score': 0.6,
   'total_cost_usd': 0.2019591,
   'average_turns': 5.333333333333333},
  'code': {'total_samples': 30,
   'total_attempted': 30,
   'errors': 0,
   'correct': 16,
   'accuracy': 0.5333333333333333,
   'average_score': 0.5333333333333333,
   'total_cost_usd': 0.35278925,
   'average_turns': 4.766666666666667},
  'database': {'total_samples': 30,
   'total_attempted': 30,
   'errors': 0,
   'correct': 30,
   'accuracy': 1.0,
   'average_score': 1.0,
   'total_cost_usd': 0.046475249999999996,
   'average_turns': 1.1333333333333333}

In [11]:
# Load results for both runs
lic_results = read_json(baseline_lic_run_dir / "results.json")
natural_results = read_json(baseline_natural_run_dir / "results.json")

lic_metrics = read_json(baseline_lic_run_dir / "metrics.json")
# natural_metrics already loaded above

print(f"LIC results: {len(lic_results)} samples")
print(f"Natural results: {len(natural_results)} samples")

LIC results: 120 samples
Natural results: 120 samples


In [12]:
def build_per_task_stats(results: list[dict]) -> dict[str, dict]:
    """Aggregate per-task stats from a results list."""
    from collections import defaultdict

    task_data = defaultdict(lambda: {
        "scores": [],
        "costs": [],
        "turns": [],
        "user_output_tokens": [],
    })

    for r in results:
        task = r["task_name"]
        task_data[task]["scores"].append(r.get("score", 0.0) or 0.0)
        task_data[task]["costs"].append(r.get("total_cost_usd", 0.0) or 0.0)
        task_data[task]["turns"].append(r.get("num_turns", 0))
        # user sim output tokens
        user_out = r.get("usage_stats", {}).get("user", {}).get("output_tokens", 0)
        task_data[task]["user_output_tokens"].append(user_out)

    stats = {}
    for task, d in sorted(task_data.items()):
        n = len(d["scores"])
        stats[task] = {
            "n": n,
            "accuracy": np.mean(d["scores"]),
            "total_cost": sum(d["costs"]),
            "avg_turns": np.mean(d["turns"]),
            "total_user_tokens": sum(d["user_output_tokens"]),
            "avg_user_tokens": np.mean(d["user_output_tokens"]),
        }
    return stats

lic_stats = build_per_task_stats(lic_results)
natural_stats = build_per_task_stats(natural_results)

print("LIC per-task stats:")
for t, s in lic_stats.items():
    print(f"  {t}: acc={s['accuracy']:.3f}, cost=${s['total_cost']:.3f}, "
          f"avg_turns={s['avg_turns']:.1f}, total_user_tok={s['total_user_tokens']}, "
          f"avg_user_tok={s['avg_user_tokens']:.0f}")

print("\nNatural per-task stats:")
for t, s in natural_stats.items():
    print(f"  {t}: acc={s['accuracy']:.3f}, cost=${s['total_cost']:.3f}, "
          f"avg_turns={s['avg_turns']:.1f}, total_user_tok={s['total_user_tokens']}, "
          f"avg_user_tok={s['avg_user_tokens']:.0f}")

LIC per-task stats:
  actions: acc=0.167, cost=$0.188, avg_turns=5.3, total_user_tok=3401, avg_user_tok=113
  code: acc=0.267, cost=$0.624, avg_turns=6.7, total_user_tok=4924, avg_user_tok=164
  database: acc=0.967, cost=$0.170, avg_turns=2.4, total_user_tok=1067, avg_user_tok=36
  math: acc=0.300, cost=$0.397, avg_turns=5.9, total_user_tok=3807, avg_user_tok=127

Natural per-task stats:
  actions: acc=0.600, cost=$0.202, avg_turns=5.3, total_user_tok=7638, avg_user_tok=255
  code: acc=0.533, cost=$0.353, avg_turns=4.8, total_user_tok=9372, avg_user_tok=312
  database: acc=1.000, cost=$0.046, avg_turns=1.1, total_user_tok=913, avg_user_tok=30
  math: acc=0.467, cost=$0.382, avg_turns=5.9, total_user_tok=9801, avg_user_tok=327


In [13]:
# Build comparison table: task (outer) x baseline type (inner) x metrics (columns)
tasks = sorted(set(list(lic_stats.keys()) + list(natural_stats.keys())))

rows = []
for task in tasks:
    for label, stats in [("LIC", lic_stats), ("Natural", natural_stats)]:
        s = stats.get(task)
        if s is None:
            continue
        rows.append({
            "task": task,
            "baseline": label,
            "accuracy": s["accuracy"],
            "total_cost": s["total_cost"],
            "avg_turns": s["avg_turns"],
            "avg_user_sim_tokens": s["avg_user_tokens"],
        })

df = pd.DataFrame(rows)
df = df.set_index(["task", "baseline"])

# Format for display
styled = df.style.format({
    "accuracy": "{:.1%}",
    "total_cost": "${:.3f}",
    "avg_turns": "{:.1f}",
    "avg_user_sim_tokens": "{:.0f}",
})
styled

In [14]:
# Add overall aggregates
def overall_stats(results):
    scores = [r.get("score", 0.0) or 0.0 for r in results]
    costs = [r.get("total_cost_usd", 0.0) or 0.0 for r in results]
    turns = [r.get("num_turns", 0) for r in results]
    user_toks = [r.get("usage_stats", {}).get("user", {}).get("output_tokens", 0) for r in results]
    return {
        "accuracy": np.mean(scores),
        "total_cost": sum(costs),
        "avg_turns": np.mean(turns),
        "avg_user_sim_tokens": np.mean(user_toks),
    }

overall_rows = rows.copy()
for label, results in [("LIC", lic_results), ("Natural", natural_results)]:
    s = overall_stats(results)
    overall_rows.append({"task": "OVERALL", "baseline": label, **s})

df_full = pd.DataFrame(overall_rows).set_index(["task", "baseline"])

styled_full = df_full.style.format({
    "accuracy": "{:.1%}",
    "total_cost": "${:.3f}",
    "avg_turns": "{:.1f}",
    "avg_user_sim_tokens": "{:.0f}",
})
styled_full

In [15]:
print(df_full.to_markdown())

|                         |   accuracy |   total_cost |   avg_turns |   avg_user_sim_tokens |
|:------------------------|-----------:|-------------:|------------:|----------------------:|
| ('actions', 'LIC')      |   0.166667 |    0.187539  |     5.33333 |              113.367  |
| ('actions', 'Natural')  |   0.6      |    0.201959  |     5.33333 |              254.6    |
| ('code', 'LIC')         |   0.266667 |    0.623529  |     6.73333 |              164.133  |
| ('code', 'Natural')     |   0.533333 |    0.352789  |     4.76667 |              312.4    |
| ('database', 'LIC')     |   0.966667 |    0.169924  |     2.43333 |               35.5667 |
| ('database', 'Natural') |   1        |    0.0464752 |     1.13333 |               30.4333 |
| ('math', 'LIC')         |   0.3      |    0.396663  |     5.9     |              126.9    |
| ('math', 'Natural')     |   0.466667 |    0.382418  |     5.86667 |              326.7    |
| ('OVERALL', 'LIC')      |   0.425    |    1.37765   |     

In [16]:
type(styled_full)

pandas.io.formats.style.Styler

In [17]:

# Token count analysis: single-turn (full_spec_q) vs. all shards concatenated
# Also: avg tokens per shard, and avg tokens per user sim message in LiC run
import tiktoken
from collections import defaultdict

enc = tiktoken.get_encoding("cl100k_base")

t30d = read_json(DATA_DIR / "t30d.json")

task_tokens = defaultdict(lambda: {"single_turn": [], "all_shards": [], "per_shard": []})

for item in t30d:
    task = item.get("task")
    full_spec = item.get("full_spec_q") or ""
    shards = item.get("shards") or []

    single_toks = len(enc.encode(full_spec))
    shard_texts = [
        s.get("shard", "") if isinstance(s, dict) else str(s) for s in shards
    ]
    shard_tok_counts = [len(enc.encode(t)) for t in shard_texts]
    shard_toks = sum(shard_tok_counts)

    task_tokens[task]["single_turn"].append(single_toks)
    task_tokens[task]["all_shards"].append(shard_toks)
    task_tokens[task]["per_shard"].extend(shard_tok_counts)  # individual shard lengths

# Avg tokens per user sim message in LiC run (output_tokens / num_requests per sample)
lic_user_msg_tokens = defaultdict(list)
for r in lic_results:
    task = r["task_name"]
    u = r.get("usage_stats", {}).get("user", {})
    out_toks = u.get("output_tokens", 0)
    n_req = u.get("num_requests", 0)
    if n_req > 0:
        lic_user_msg_tokens[task].append(out_toks / n_req)

rows = []
for task in sorted(task_tokens):
    st = task_tokens[task]["single_turn"]
    sh = task_tokens[task]["all_shards"]
    ps = task_tokens[task]["per_shard"]
    um = lic_user_msg_tokens.get(task, [])
    rows.append({
        "task": task,
        "n": len(st),
        "single_turn_avg_tok": sum(st) / len(st),
        "single_turn_total_tok": sum(st),
        "all_shards_avg_tok": sum(sh) / len(sh),
        "all_shards_total_tok": sum(sh),
        "avg_tok_per_shard": sum(ps) / len(ps) if ps else float("nan"),
        "lic_avg_tok_per_user_msg": sum(um) / len(um) if um else float("nan"),
    })

# Overall row
all_st = [v for d in task_tokens.values() for v in d["single_turn"]]
all_sh = [v for d in task_tokens.values() for v in d["all_shards"]]
all_ps = [v for d in task_tokens.values() for v in d["per_shard"]]
all_um = [v for vs in lic_user_msg_tokens.values() for v in vs]
rows.append({
    "task": "OVERALL",
    "n": len(all_st),
    "single_turn_avg_tok": sum(all_st) / len(all_st),
    "single_turn_total_tok": sum(all_st),
    "all_shards_avg_tok": sum(all_sh) / len(all_sh),
    "all_shards_total_tok": sum(all_sh),
    "avg_tok_per_shard": sum(all_ps) / len(all_ps),
    "lic_avg_tok_per_user_msg": sum(all_um) / len(all_um),
})

df_tokens = pd.DataFrame(rows).set_index("task")
df_tokens.style.format({
    "single_turn_avg_tok": "{:.0f}",
    "single_turn_total_tok": "{:,}",
    "all_shards_avg_tok": "{:.0f}",
    "all_shards_total_tok": "{:,}",
    "avg_tok_per_shard": "{:.1f}",
    "lic_avg_tok_per_user_msg": "{:.1f}",
})


,n,single_turn_avg_tok,single_turn_total_tok,all_shards_avg_tok,all_shards_total_tok,avg_tok_per_shard,lic_avg_tok_per_user_msg
task,,,,,,,
actions,30,61,"1,826",64,"1,924",11.5,26.2
code,30,245,"7,340",87,"2,619",12.1,28.4
database,30,18,543,38,"1,154",8.7,25.6
math,30,68,"2,030",70,"2,104",11.9,25.5
OVERALL,120,98,"11,739",65,"7,801",11.3,26.4
